In [1]:
# ID : 고유 식별자
# Marital Status : 혼인 여부
# Application mode : 지원 방식
# Course : 지원한 학과
# Daytime/evening attendance : 주간/야간 수업 구분
# Previous qualification : 이전 학력
# Nationality : 국적
# Mother’s qualification : 어머니의 학력
# Father’s qualification : 아버지의 학력
# Mother’s occupation : 어머니의 직업
# Father’s occupation : 아버지의 직업
# Educational special needs : 교육적 특수 요구 여부
# Gender : 성별
# Curricular units 1st sem (credited) : 1학기 인정 학점 수
# Curricular units 1st sem (enrolled) : 1학기 수강 과목 수
# Curricular units 1st sem (evaluations) : 1학기 평가 횟수
# Curricular units 1st sem (approved) : 1학기 통과 과목 수
# Target : Graduate(졸업)	/ Dropout(중퇴)	/ Enrolled(재학중), 학업 결과 (분류 대상 변수)

In [2]:
import pandas as pd

In [3]:
train = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_2_train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_2_test.csv')


In [4]:
train.head(1)

,level_0,ID,Marital Status,Application mode,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Educational special needs,Gender,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Target
0,0,3949,1_,43_,9147_,1_,1_,1_,1_,19_,9_,10_,0_,0_,0,5,9,5,Graduate


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 19 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   level_0                                 309 non-null    int64 
 1   ID                                      309 non-null    int64 
 2   Marital Status                          309 non-null    object
 3   Application mode                        309 non-null    object
 4   Course                                  309 non-null    object
 5   Daytime/evening attendance              309 non-null    object
 6   Previous qualification                  309 non-null    object
 7   Nacionality                             309 non-null    object
 8   Mother's qualification                  309 non-null    object
 9   Father's qualification                  309 non-null    object
 10  Mother's occupation                     309 non-null    object
 11  Father

In [12]:
X = train.drop(columns=['level_0','ID','Target'], axis=1) # columns, axis=1 중 하나면 설정해도 된다
y = train['Target']

X_test = test.drop(columns=['level_0','ID','Target'])

y.value_counts()

,count
Target,
Graduate,154
Dropout,85
Enrolled,70


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, stratify=y)

X_train.shape

(216, 16)

In [21]:
cat_columns = X_train.select_dtypes('object').columns
num_columns = X_train.select_dtypes('number').columns
cat_columns

Index(['Marital Status', 'Application mode', 'Course',
       'Daytime/evening attendance', 'Previous qualification', 'Nacionality',
       'Mother's qualification', 'Father's qualification',
       'Mother's occupation', 'Father's occupation',
       'Educational special needs', 'Gender'],
      dtype='object')

In [17]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').set_output(transform='pandas')

In [23]:
X_train_en = encoder.fit_transform(X_train[cat_columns])
X_val_en = encoder.transform(X_val[cat_columns])
X_test_en = encoder.transform(X_test[cat_columns])

# X_train_en

X_train_pre = pd.concat([X_train_en, X_train[num_columns]], axis=1)
X_val_pre = pd.concat([X_val_en, X_val[num_columns]], axis=1)
X_test_pre = pd.concat([X_test_en, X_test[num_columns]], axis=1)

In [29]:
# RandomForestClassifier -> 하이퍼 파라미터값을 비교(여러 입력값 변경해가며 비교)

from sklearn.ensemble import RandomForestClassifier

models = {
    'RF_100' : RandomForestClassifier(n_estimators=100) ,
    'RF_200' : RandomForestClassifier(n_estimators=200) ,
    'RF_200_depth10' : RandomForestClassifier(n_estimators=200, max_depth=10) ,
    'RF_200_depth20' : RandomForestClassifier(n_estimators=200, max_depth=20) ,
    'RF_cw' : RandomForestClassifier(n_estimators=200, class_weight='balanced') ,
} # 5개의 키:값

from sklearn.metrics import accuracy_score

result = [] # models 5개의 평가지표를 저장

for k, v in models.items(): # k:이름, v:이름별 모델
  # print(k)
  v.fit(X_train_pre, y_train) # 학습
  pred_val = v.predict(X_val_pre)
  accuracy = accuracy_score(y_val, pred_val)
  result.append({
      'model_name' : k,
      'accuracy' : accuracy
  })

result

[{'model_name': 'RF_100', 'accuracy': 0.6451612903225806},
 {'model_name': 'RF_200', 'accuracy': 0.6774193548387096},
 {'model_name': 'RF_200_depth10', 'accuracy': 0.6559139784946236},
 {'model_name': 'RF_200_depth20', 'accuracy': 0.6451612903225806},
 {'model_name': 'RF_cw', 'accuracy': 0.6666666666666666}]

In [30]:
result_df = pd.DataFrame(result)
result_df

,model_name,accuracy
0,RF_100,0.645161
1,RF_200,0.677419
2,RF_200_depth10,0.655914
3,RF_200_depth20,0.645161
4,RF_cw,0.666667


In [31]:
result_df_sort = result_df.sort_values('accuracy', ascending=False)
result_df_sort

,model_name,accuracy
1,RF_200,0.677419
4,RF_cw,0.666667
2,RF_200_depth10,0.655914
0,RF_100,0.645161
3,RF_200_depth20,0.645161


In [34]:
# 모델 이름

best_model_name = result_df_sort.iloc[0]['model_name'] # result_df_sort.iloc[0, 0]
best_model_name

'RF_200'

In [36]:
best_model = models[best_model_name] # 파이썬 사전타입

In [37]:
pred_test = best_model.predict(X_test_pre)
pred_test

array(['Graduate', 'Graduate', 'Graduate', 'Dropout', 'Graduate',
       'Dropout', 'Dropout', 'Graduate', 'Graduate', 'Graduate',
       'Graduate', 'Graduate', 'Graduate', 'Dropout', 'Graduate',
       'Graduate', 'Enrolled', 'Dropout', 'Graduate', 'Graduate',
       'Dropout', 'Graduate', 'Graduate', 'Graduate', 'Enrolled',
       'Graduate', 'Graduate', 'Graduate', 'Graduate', 'Graduate',
       'Graduate', 'Dropout', 'Graduate', 'Enrolled', 'Graduate',
       'Dropout', 'Graduate', 'Dropout', 'Graduate', 'Dropout',
       'Graduate', 'Graduate', 'Dropout', 'Graduate', 'Enrolled',
       'Graduate', 'Dropout', 'Graduate', 'Dropout', 'Graduate',
       'Graduate', 'Enrolled', 'Graduate', 'Graduate', 'Dropout',
       'Graduate', 'Dropout', 'Dropout', 'Graduate', 'Enrolled',
       'Enrolled', 'Graduate', 'Graduate', 'Graduate', 'Graduate',
       'Graduate', 'Graduate', 'Enrolled', 'Enrolled', 'Dropout',
       'Dropout', 'Graduate', 'Dropout', 'Graduate', 'Graduate',
       'Gradua